# SEAMLESS Reel Generator — Simple
Image to Video (Stable Video Diffusion)

**Как работает:**
1. Загружаешь фото с CI-танцем
2. SVD оживляет его в 3-сек видео
3. Скачиваешь

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**

Нажми **Run All** и подожди ~3 мин.

In [ ]:
# Setup
!pip install -q torch torchvision accelerate diffusers transformers pillow imageio imageio-ffmpeg
print('Setup done')

In [ ]:
# Upload your CI photo
from google.colab import files
print('Upload a photo of CI dance (JPG or PNG):')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
print(f'Loaded: {img_path}')

In [ ]:
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image

# Load & prepare image
img = Image.open(img_path).convert('RGB')
# Resize to SVD-friendly size
img = img.resize((1024, 576))
img.save('/content/input.png')
display(img)
print('Loading SVD model...')

pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16'
)
pipe.to('cuda')
pipe.enable_model_cpu_offload()
print('Model loaded!')

# Generate
print('Generating video...')
result = pipe(
    img,
    decode_chunk_size=8,
    num_frames=25,
    motion_bucket_id=127,
    noise_aug_strength=0.02
)

path = '/content/seamless_reel.mp4'
export_to_video(result.frames[0], path, fps=7)
print('Done!')

In [ ]:
# Preview
from IPython.display import Video
Video('/content/seamless_reel.mp4', embed=True)

In [ ]:
# Download
from google.colab import files
files.download('/content/seamless_reel.mp4')